In [0]:
incoming_path = "/Volumes/bmw_vehicle_intelligence/bronze/raw_data/incoming"

dbutils.fs.mkdirs(incoming_path)

print("Incoming folder created successfully!")

In [0]:
incoming_path = "/Volumes/bmw_vehicle_intelligence/bronze/raw_data/incoming"

files = dbutils.fs.ls(incoming_path)

for file in files:
    print(file.name, file.path, file.size)

In [0]:
file_path = "/Volumes/bmw_vehicle_intelligence/bronze/raw_data/incoming/bmw_vehicle_new.csv"

new_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(file_path)
)

print("New file records:", new_df.count())

display(new_df)

In [0]:
# file_path = "/Volumes/bmw_vehicle_intelligence/bronze/raw_data/incoming/bmw_vehicle_new.csv"

# new_df = (
#     spark.read
#     .option("header", "true")
#     .option("inferSchema", "true")
#     .csv(file_path)
# )

# print("Incoming Records:", new_df.count())

# display(new_df)

In [0]:
dbutils.fs.ls("/Volumes/bmw_vehicle_intelligence/bronze/raw_data")

In [0]:
source_path = "/Volumes/bmw_vehicle_intelligence/bronze/raw_data/bmw_vehicle_new.csv"

destination_dir = "/Volumes/bmw_vehicle_intelligence/bronze/raw_data/incoming"
destination_path = destination_dir + "/bmw_vehicle_new.csv"

existing_files = [f.name for f in dbutils.fs.ls(destination_dir)]

if "bmw_vehicle_new.csv" in existing_files:
    print("File already exists in the incoming folder — skipping move.")
else:
    dbutils.fs.mv(source_path, destination_path)
    print("File moved successfully!")

In [0]:
file_path = "/Volumes/bmw_vehicle_intelligence/bronze/raw_data/incoming/bmw_vehicle_new.csv"

new_df = (
    spark.read
    .option("header", "true")
    .option("inferSchema", "true")
    .csv(file_path)
)

print("Incoming Records:", new_df.count())

display(new_df)

In [0]:
from pyspark.sql.functions import *
from delta.tables import DeltaTable

merge_df = (
    new_df
    .withColumn("vehicle_id", col("vehicle_id").cast("int"))
    .withColumn("customer_id", col("customer_id").cast("int"))
    .withColumn("model_year", col("model_year").cast("int"))
    .withColumn("purchase_date", to_date("purchase_date"))
    .withColumn("vin", upper(trim(col("vin"))))
    .withColumn("model", trim(col("model")))
    .withColumn("fuel_type", upper(trim(col("fuel_type"))))
    .withColumn("processed_at", current_timestamp())
)


target = DeltaTable.forName(
    spark,
    "bmw_vehicle_intelligence.silver.bmw_vehicle"
)


(
    target.alias("target")
    .merge(
        merge_df.alias("source"),
        "target.vehicle_id = source.vehicle_id"
    )
    .whenMatchedUpdateAll()
    .whenNotMatchedInsertAll()
    .execute()
)

print("Delta MERGE completed successfully!")

In [0]:
from pyspark.sql.functions import *

silver_df = spark.table(
    "bmw_vehicle_intelligence.silver.bmw_vehicle"
)

print("Total Silver Records:", silver_df.count())

# Check inserted and updated records
display(
    silver_df
    .filter(col("vehicle_id").isin(10001, 10004, 10005))
    .select(
        "vehicle_id",
        "customer_id",
        "model",
        "fuel_type",
        "processed_at"
    )
)

In [0]:
# 1. Vehicle summary
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.vehicle_summary AS
SELECT
    model,
    fuel_type,
    COUNT(*) AS total_vehicles,
    ROUND(AVG(model_year), 2) AS average_model_year
FROM bmw_vehicle_intelligence.silver.bmw_vehicle
GROUP BY model, fuel_type
""")

# 2. Model summary
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.model_summary AS
SELECT
    model,
    COUNT(*) AS total_vehicles,
    MIN(model_year) AS oldest_model_year,
    MAX(model_year) AS newest_model_year
FROM bmw_vehicle_intelligence.silver.bmw_vehicle
GROUP BY model
""")

# 3. Fuel summary
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.fuel_summary AS
SELECT
    fuel_type,
    COUNT(*) AS total_vehicles,
    ROUND(
        COUNT(*) * 100.0 / SUM(COUNT(*)) OVER (), 2
    ) AS percentage
FROM bmw_vehicle_intelligence.silver.bmw_vehicle
GROUP BY fuel_type
""")

# 4. Purchase year summary
spark.sql("""
CREATE OR REPLACE TABLE bmw_vehicle_intelligence.gold.purchase_year_summary AS
SELECT
    YEAR(purchase_date) AS purchase_year,
    COUNT(*) AS total_purchases
FROM bmw_vehicle_intelligence.silver.bmw_vehicle
GROUP BY YEAR(purchase_date)
""")

print("All Gold vehicle summaries refreshed successfully!")

In [0]:
%sql
SELECT COUNT(*) AS total_vehicles
FROM bmw_vehicle_intelligence.silver.bmw_vehicle;

In [0]:
%sql
SELECT SUM(total_vehicles) AS gold_total_vehicles
FROM bmw_vehicle_intelligence.gold.vehicle_summary;

In [0]:
%sql
SELECT *
FROM bmw_vehicle_intelligence.gold.model_summary
ORDER BY total_vehicles DESC;

In [0]:
%sql
SELECT *
FROM bmw_vehicle_intelligence.gold.fuel_summary
ORDER BY total_vehicles DESC;

In [0]:
%sql
SELECT *
FROM bmw_vehicle_intelligence.gold.purchase_year_summary
ORDER BY purchase_year DESC;

In [0]:
%sql
OPTIMIZE bmw_vehicle_intelligence.silver.bmw_vehicle
ZORDER BY (vehicle_id);

In [0]:
%sql

CREATE TABLE IF NOT EXISTS bmw_vehicle_intelligence.silver.bmw_vehicle_history
AS
SELECT
    *,
    current_timestamp() AS effective_start_date,
    CAST(NULL AS TIMESTAMP) AS effective_end_date,
    TRUE AS is_current
FROM bmw_vehicle_intelligence.silver.bmw_vehicle;

In [0]:
%sql

SELECT
    COUNT(*) AS total_records,
    COUNT(CASE WHEN is_current = TRUE THEN 1 END) AS current_records
FROM bmw_vehicle_intelligence.silver.bmw_vehicle_history;

In [0]:
%sql

SELECT
    vehicle_id,
    model,
    fuel_type,
    effective_start_date,
    effective_end_date,
    is_current
FROM bmw_vehicle_intelligence.silver.bmw_vehicle_history
WHERE vehicle_id = 10001;

In [0]:
%sql

CREATE OR REPLACE TEMP VIEW vehicle_changes AS
SELECT
    vehicle_id,
    customer_id,
    vin,
    'BMW X5 M Sport Updated' AS model,
    model_year,
    fuel_type,
    purchase_date,
    processed_at
FROM bmw_vehicle_intelligence.silver.bmw_vehicle
WHERE vehicle_id = 10001;

In [0]:
%sql

UPDATE bmw_vehicle_intelligence.silver.bmw_vehicle_history
SET
    effective_end_date = current_timestamp(),
    is_current = FALSE
WHERE vehicle_id = 10001
  AND is_current = TRUE
  AND model <> 'BMW X5 M Sport Updated';

In [0]:
%sql

INSERT INTO bmw_vehicle_intelligence.silver.bmw_vehicle_history
(
    vehicle_id,
    customer_id,
    vin,
    model,
    model_year,
    fuel_type,
    purchase_date,
    processed_at,
    effective_start_date,
    effective_end_date,
    is_current
)
SELECT
    vehicle_id,
    customer_id,
    vin,
    model,
    model_year,
    fuel_type,
    purchase_date,
    processed_at,
    current_timestamp(),
    CAST(NULL AS TIMESTAMP),
    TRUE
FROM vehicle_changes;

In [0]:
%sql

SELECT
    vehicle_id,
    model,
    effective_start_date,
    effective_end_date,
    is_current
FROM bmw_vehicle_intelligence.silver.bmw_vehicle_history
WHERE vehicle_id = 10001
ORDER BY effective_start_date;